In [27]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-2uiuzxe8
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-2uiuzxe8
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [28]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b


In [29]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [30]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b


In [31]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};


namespace gpu {


/**
 * Constructs a DenseGraph from an input edge list of m edges.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    if (th_id < m) {
        edge_t e = edge_list[th_id];

        g.adjacencyMatrix[e.x * g.n + e.y] = 1; //x axis
        g.adjacencyMatrix[e.y * g.n + e.x] = 1; //y axis - duplicat ewrite lol
    }

    return;
}

/**
  * Repopulates the adjacency matrix as a new graph that represents
  * the two-hop neighbourhood of input graph g
  */

constexpr int TILE = 32;

__global__
void two_hop_reachability( DenseGraph g, node_t * out )    // launch with block = (TILE, TILE)

{   //y * TILE + x

    __shared__ node_t As[ TILE ][ TILE ];   // a 32x32 temp storage of A, for the ROWS this block needs (output rows and intermediate vertices needed for multiplication)
    __shared__ node_t Bs[ TILE ][ TILE ];   // a 32x32 temp storage of A, for the COLUMNS this block needs (intermediate vertices and output columns needed for multiplication)

    std::size_t const n   = g.n;  //elements


    // per thread output cell: block position * tile size + position inside the block
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    node_t paths = 0;   // (init) number of 2-step paths

    for ( std::size_t t = 0; t < n; t += TILE )          // walk across k in chunks of 32
    {
        std::size_t const a_col = t + threadIdx.x;
        std::size_t const b_row = t + threadIdx.y;

        // cooperative load: each thread fetches one element of each tile - ADJ TO SHARED
        // out-of-range threads store 0 instead of returning,
        // because every thread must reach __syncthreads()

        if (row < n && a_col < n)
        {
            As[threadIdx.y][threadIdx.x] = g.adjacencyMatrix[row * n + a_col];
        }
        else
        {
            As[threadIdx.y][threadIdx.x] = 0;
        }
        if (b_row < n && col < n)
        {
            Bs[threadIdx.y][threadIdx.x] = g.adjacencyMatrix[b_row * n + col];
        }
        else
        {
            Bs[threadIdx.y][threadIdx.x] = 0;
        }
        __syncthreads();                                  // tiles fully loaded

        for ( int k = 0; k < TILE; ++k )
            paths += As[ threadIdx.y ][ k ] * Bs[ k ][ threadIdx.x ]; //paths counts how many middle nodes k connect row to col.
        __syncthreads();                                  // everyone done before tiles are overwritten
    }

    if (row < n && col < n)
    {
        if (row != col && paths > 0) // two step exists and not diagonal (n, n)
        {
            out[row * n + col] = 1;
        }
        else
        {
            out[row * n + col] = 0;
        }
    }                                     // drop diagonal, clamp to 0 and 1s
}


} // namespace gpu
} // namespace a2
} // namespace csc485b

In [32]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};


namespace gpu {

/**
 * Constructs a SparseGraph from an input edge list of m edges.
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 */


__global__
void histogram( SparseGraph g, edge_t const * edge_list, std::size_t m )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;    //glob id

    if (th_id < m){

        edge_t e = edge_list[th_id];  // thread per edge format

        atomicAdd(&g.neighbours_start_at[e.x], 1);  //increment source node neighbour count, atomic to stop race

        //#EXCLUSIVE SCAN (PREFIX SUM) GIVES START POINTs

    }

    return;
}

__global__
void exclusive_scan( node_t * data, std::size_t len )
{
    __shared__ node_t sums[ 1024 ];

    std::size_t const tid     = threadIdx.x;        //thread ids
    std::size_t const T     = blockDim.x;           //total threads
    std::size_t const chunk = ( len + T - 1 ) / T;  //thread work size



    std::size_t begin, end;

    if (tid * chunk < len) { begin = tid * chunk; } //in bounds, chunk starts here
    else { begin = len; } //if outside, point to nothing (end)

    if (begin + chunk < len) { end = begin + chunk; } //in bounds, chunk stops here (exclusive)
    else { end = len; } //cap at array length


    // 1 - threads sum their chunk and write total

    node_t total = 0;
    for ( std::size_t i = begin; i < end; ++i ){

        total += data[ i ];
    }
    sums[ tid ] = total;
    __syncthreads();      //all chunk totals written first


    // 2 - scan the chunk totals; each round adds the value 'offset' places to the left, offset doubles, log time

    for ( std::size_t offset = 1; offset < T; offset <<= 1 )
    {
        node_t v;
        if (tid >= offset) { v = sums[tid - offset]; }  // read old neigbour
        else { v = 0; }   // nothing = head = 0

        __syncthreads(); // All reads done
        sums[ tid ] += v;   //add prev offset to current partial sum
        __syncthreads();  // all writes done
    }

    //sums now holds total of each tid chunk, inclusive

    node_t running;
    if (tid == 0) { running = 0; }  //first chunk is 0
    else { running = sums[tid - 1]; };  //otherwise assign value of everything beforehand

    for ( std::size_t i = begin; i < end; ++i )
    {
        node_t const d = data[ i ]; //read node degree
        data[ i ] = running;    //overwrite with running
        running += d; //increment to next
    }
}

__global__
void build_graph( SparseGraph g, edge_t const * edge_list, std::size_t m, node_t * inserted_count )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    if ( th_id < m )
    {
        edge_t const e = edge_list[ th_id ];

        int pos = atomicAdd(&inserted_count[e.x], 1); //prev neighbour count tallied

        g.neighbours[g.neighbours_start_at[e.x] + pos] = e.y; // thread finds its spot in neigbour with offset + individual x pos value
    }
}


/**
  * Repopulates the adjacency lists as a new graph that represents
  * the two-hop neighbourhood of input graph g
  */
__global__
void two_hop_reachability( SparseGraph g )
{
    // IMPLEMENT ME!
    // algorithm unknown
    return;
}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [33]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <chrono>   // for timing
#include <iostream> // std::cout, std::endl
#include <iterator> // std::ostream_iterator
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "data_generator.h"
#include "data_types.h"

/**
 * Runs timing tests on a CUDA graph implementation.
 * Consists of independently constructing the graph and then
 * modifying it to its two-hop neighbourhood.
 */
template < typename DeviceGraph >
void run( DeviceGraph g, csc485b::a2::edge_t const * d_edges, std::size_t m )
{
    csc485b::a2::node_t * d_out = nullptr;  //Allocation for dense two hop final output
    if constexpr ( std::is_same_v< DeviceGraph, csc485b::a2::DenseGraph > )
        cudaMalloc( (void**)&d_out, sizeof( *d_out ) * g.n * g.n );

    cudaDeviceSynchronize();
    auto const build_start = std::chrono::high_resolution_clock::now();

    // BUILD LAUNCH!!!!!

    std::size_t block_size = 256;
    std::size_t const num_blocks = ( m + block_size - 1 ) / block_size;

    if constexpr ( std::is_same_v< DeviceGraph, csc485b::a2::SparseGraph > )
    {

        csc485b::a2::node_t * d_inserted_count; // temp counter for nnext neighbour position in CSR

        cudaMalloc((void**)&d_inserted_count, sizeof(csc485b::a2::node_t) * g.n);
        cudaMemset(d_inserted_count, 0, sizeof(csc485b::a2::node_t) * g.n);


        csc485b::a2::gpu::histogram<<< num_blocks, block_size >>>(g, d_edges, m);

        csc485b::a2::gpu::exclusive_scan<<< 1, 1024 >>>(g.neighbours_start_at, g.n);

        csc485b::a2::gpu::build_graph<<< num_blocks, block_size >>>(g, d_edges, m, d_inserted_count);


        cudaFree(d_inserted_count );
    }
    else
    {
        // dense
        csc485b::a2::gpu::build_graph<<< num_blocks, block_size >>>( g, d_edges, m );
    }

    cudaDeviceSynchronize();
    auto const reachability_start = std::chrono::high_resolution_clock::now();

    // neither does this!

    if constexpr ( std::is_same_v< DeviceGraph, csc485b::a2::DenseGraph > )
    {
        dim3 const block( csc485b::a2::gpu::TILE, csc485b::a2::gpu::TILE );
        dim3 const grid( ( g.n + csc485b::a2::gpu::TILE - 1 ) / csc485b::a2::gpu::TILE,
                        ( g.n + csc485b::a2::gpu::TILE - 1 ) / csc485b::a2::gpu::TILE );

        csc485b::a2::gpu::two_hop_reachability<<< grid, block >>>( g, d_out );

        // copy result back into graph
        cudaMemcpy( g.adjacencyMatrix, d_out, sizeof( csc485b::a2::node_t ) * g.n * g.n,
                    cudaMemcpyDeviceToDevice );

    }
    else
    {
        // sparse: your existing stub launch, unchanged
        csc485b::a2::gpu::two_hop_reachability<<< 1, 1 >>>( g );
    }

    cudaDeviceSynchronize();
    auto const end = std::chrono::high_resolution_clock::now();

    std::cout << "Build time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(reachability_start - build_start).count()
              << " us"
              << std::endl;

    std::cout << "Reachability time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(end - reachability_start).count()
              << " us"
              << std::endl;


    cudaFree( d_out );
}

/**
 * Allocates space for a dense graph and then runs the test code on it.
 */
void run_dense( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m)
{
    using namespace csc485b;

    // allocate device DenseGraph
    a2::node_t * d_matrix;
    cudaMalloc((void**)&d_matrix, sizeof(a2::node_t) * n * n);
    cudaMemset(d_matrix, 0, sizeof( a2::node_t) * n * n);
    a2::DenseGraph d_dg{n, d_matrix};

    run(d_dg, d_edges, m);

    // check output?


    std::vector< a2::node_t > host_matrix( d_dg.matrix_size() );
    a2::DenseGraph dg{ n, host_matrix.data() };
    cudaMemcpy( dg.adjacencyMatrix, d_dg.adjacencyMatrix, sizeof( a2::node_t ) * d_dg.matrix_size(), cudaMemcpyDeviceToHost );
    std::copy( host_matrix.cbegin(), host_matrix.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );



    // clean up
    cudaFree( d_matrix );
}

/**
 * Allocates space for a sparse graph and then runs the test code on it.
 */
void run_sparse( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m )
{
    using namespace csc485b;

    // allocate device SparseGraph
    a2::node_t * d_offsets, * d_neighbours;
    cudaMalloc( (void**)&d_offsets,    sizeof( a2::node_t ) * n );
    cudaMalloc( (void**)&d_neighbours, sizeof( a2::node_t ) * m );
    cudaMemset( d_offsets, 0, sizeof( a2::node_t ) * n );
    a2::SparseGraph d_sg{ n, m, d_offsets, d_neighbours };

    run( d_sg, d_edges, m );




    // PAINFUL SPARSE OUTPUT WORK!!


    std::vector< a2::node_t > host_offsets( n ), host_neighbours( m );
    cudaMemcpy( host_offsets.data(),    d_sg.neighbours_start_at,    sizeof( a2::node_t ) * n, cudaMemcpyDeviceToHost );
    cudaMemcpy( host_neighbours.data(), d_sg.neighbours, sizeof( a2::node_t ) * m, cudaMemcpyDeviceToHost );

    std::cout << "\noffsets: ";
    std::copy( host_offsets.cbegin(), host_offsets.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    std::cout << "\nneighbours: ";
    std::copy( host_neighbours.cbegin(), host_neighbours.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    std::cout << std::endl;


    // clean up
    cudaFree( d_neighbours );
    cudaFree( d_offsets );
}

int main()
{
    using namespace csc485b;

    // Create input
    std::size_t constexpr n = 68;
    std::size_t constexpr expected_degree = 16;   //DEFAULT = n >> 1

    a2::edge_list_t const graph = a2::generate_graph( n, n * expected_degree );
    std::size_t const m = graph.size();

    // lazily echo out input graph



    for( auto const& e : graph )
    {
        std::cout << "(" << e.x << "," << e.y << ") ";
    }


    // allocate and memcpy input to device
    a2::edge_t * d_edges;
    cudaMalloc( (void**)&d_edges, sizeof( a2::edge_t ) * m );
    cudaMemcpyAsync( d_edges, graph.data(), sizeof( a2::edge_t ) * m, cudaMemcpyHostToDevice );

    // run your code!
    run_dense ( d_edges, n, m);
    run_sparse( d_edges, n, m );

    return EXIT_SUCCESS;
}

In [34]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

(0,2) (2,0) (0,3) (3,0) (0,5) (5,0) (0,6) (6,0) (0,9) (9,0) (0,11) (11,0) (0,14) (14,0) (0,16) (16,0) (0,20) (20,0) (0,24) (24,0) (0,26) (26,0) (0,27) (27,0) (0,28) (28,0) (0,32) (32,0) (0,35) (35,0) (0,36) (36,0) (0,43) (43,0) (0,50) (50,0) (0,51) (51,0) (0,60) (60,0) (0,67) (67,0) (1,2) (2,1) (1,8) (8,1) (1,15) (15,1) (1,17) (17,1) (1,23) (23,1) (1,25) (25,1) (1,27) (27,1) (1,34) (34,1) (1,45) (45,1) (1,47) (47,1) (1,48) (48,1) (1,51) (51,1) (1,55) (55,1) (1,60) (60,1) (1,64) (64,1) (2,4) (4,2) (2,5) (5,2) (2,6) (6,2) (2,9) (9,2) (2,10) (10,2) (2,12) (12,2) (2,16) (16,2) (2,17) (17,2) (2,23) (23,2) (2,25) (25,2) (2,27) (27,2) (2,28) (28,2) (2,30) (30,2) (2,31) (31,2) (2,39) (39,2) (2,43) (43,2) (2,45) (45,2) (2,48) (48,2) (2,50) (50,2) (2,52) (52,2) (2,55) (55,2) (2,57) (57,2) (3,7) (7,3) (3,11) (11,3) (3,12) (12,3) (3,13) (13,3) (3,15) (15,3) (3,24) (24,3) (3,27) (27,3) (3,29) (29,3) (3,33) (33,3) (3,36) (36,3) (3,45) (45,3) (3,64) (64,3) (3,67) (67,3) (4,5) (5,4) (4,11) (11,4) (4,1